In [ ]:
import json
import os
import re
from PIL import Image

INPUT_FILE = "MPDocBench.json"
OUTPUT_PATH = "./markdown/glm_ocr"

MD_PATH = OUTPUT_PATH + "_md"
os.makedirs(MD_PATH, exist_ok=True)

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
    raw_data = {}
    for item in data:
        page_info = item["page_info"]
        images_list = page_info["images_list"]
        annotations_list = page_info["annotations_list"]
        image_path = page_info["image_path"]
        pdf_name = os.path.splitext(image_path)[0]
        if pdf_name not in raw_data:
            raw_data[pdf_name] = []
            page_id = 0
            for img, ann in zip(images_list, annotations_list):
                raw_data[pdf_name].append((page_id, img, ann))
                page_id += 1
        else:
            print(f"Warning: duplicate pdf_name {pdf_name} found. Skipping.")
            continue
    data = raw_data

def normalized_bbox_2_1000(bbox_list, width, height):
    x1, y1, x2, y2 = bbox_list
    return list(map(int, [x1/width*1000, y1/height*1000, x2/width*1000, y2/height*1000]))

for pdf_name in data:
    md_dir = os.path.join(OUTPUT_PATH, pdf_name)
    md_path = os.path.join(md_dir, f"{pdf_name}.md")
    json_path = os.path.join(md_dir, f"{pdf_name}.json")

    if os.path.exists(md_path) and os.path.exists(json_path):
        try:
            with open(md_path, 'r', encoding='utf-8') as f:
                md_content = f.read()
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
        except Exception as e:
            print(f"Error reading files for {pdf_name}: {e}")
            continue

        images = {}
        index = 0
        for page_idx, page_dets in enumerate(json_data):
            layout_vis_img_path = os.path.join(md_dir, "layout_vis", f"{pdf_name}_page{page_idx}.jpg")
            
            # 增加图片存在性检查，防止找不到图片报错
            if not os.path.exists(layout_vis_img_path):
                print(f"Image not found: {layout_vis_img_path}")
                continue
                
            img = Image.open(layout_vis_img_path)
            width, height = img.size
            for layout in page_dets:
                label = layout["label"]
                bbox_2d = layout["bbox_2d"]
                if label == "image":
                    image_label = f"![Image {page_idx}-{index}](imgs/cropped_page{page_idx}_idx{index}.jpg)"
                    bbox = bbox_2d # 本来就是归一化0-1000,所以不需要归一化了
                    bbox_string = f"{bbox[0]}_{bbox[1]}_{bbox[2]}_{bbox[3]}"
                    new_image_label = f"![](page{page_idx+1}_{bbox_string}.jpg)"
                    
                    if image_label in images:
                        print(f"Duplicate image label found: {pdf_name} {image_label}")
                    images[image_label] = new_image_label
                    index += 1
        
        # ---------------- 以下为新增的替换与保存逻辑 ----------------
        
        # 1. 遍历 images 字典，将 md_content 中的 key 替换为 value
        for old_label, new_label in images.items():
            if old_label in md_content:
                md_content = md_content.replace(old_label, new_label, 1)
            else:
                print(f"Warning: Label {old_label} not found in {pdf_name}.md")

        try:
            with open(os.path.join(MD_PATH, f"{pdf_name}.md"), 'w', encoding='utf-8') as f:
                f.write(md_content)                
        except Exception as e:
            print(f"Error saving modified md file for {pdf_name}: {e}")
    else:
        print(f"{pdf_name} does not exist.")